## Session 2 · Topic 5 — Reading and Writing DataFrames (Excel ⇄ CSV)

**Dataset:** `Y1_T1_2025.xlsx` — Term 1, 2025 first-year unit-attempt extract
(one row per student unit attempt, 25 columns).

In this notebook you build a simple student list for one unit, **ECO101**, and
along the way practise the two ways of moving data in and out of pandas:

- **Reading Excel** with `pd.read_excel()`
- **Writing and reading CSV** with `DataFrame.to_csv()` and `pd.read_csv()`

You will read the Excel extract, filter it to ECO101, add two new columns, remove
duplicate rows, keep only the student particulars you need, write the result to a
CSV file, and then read that CSV back in.

Replace every `# TODO` with your own code and run the cell.

### 1. Load the Excel extract

Read the **`Extract`** sheet of `data/Y1_T1_2025.xlsx` into a DataFrame called
`df`, and print its shape.

In [ ]:
import pandas as pd
from pathlib import Path

DATA_FOLDER = Path("../data")
DATA_FILE = DATA_FOLDER / "Y1_T1_2025.xlsx"

df = pd.read_excel(DATA_FILE, sheet_name="Extract")
print(df.shape)

### 2. Filter to ECO101

The extract covers three units (`ECO101`, `ECO102`, `ECO103`). Keep only the
`ECO101` rows, in a new DataFrame called `eco101`, and print how many rows that
leaves.

In [ ]:
df_eco101 = df[df["unit_code"] == "ECO101"].copy()
print(df_eco101.shape)

### 3. Add a `full_name` column

`first_name` and `family_name` are separate columns. Join them with a space into
a new `full_name` column, the same idea as the `&` text operator you used in
Excel.

In [ ]:
df_eco101["full_name"] = df_eco101["first_name"] + " " + df_eco101["family_name"]
df_eco101[["first_name", "family_name", "full_name"]].head()

### 4. Add a `result` column

Turn `unit_attempt_status` into a short result code:

- `"Passed"` → `"P"`
- `"Failed"` → `"F"`
- `"Withdrawn"` → `""` (empty — no result to report)

A dictionary plus `.map()` does this in one line.

In [ ]:
result_map = {"Passed": "P", "Failed": "F", "Withdrawn": ""}
df_eco101["result"] = df_eco101["unit_attempt_status"].map(result_map)
df_eco101["result"].value_counts(dropna=False)

### 5. Keep only the student particulars

Reduce the table to the columns you actually need — a small, clean student list:

`student_id`, `full_name`, `gender`, `program_code`, `country_of_birth`, `result`

Call the result `students`.

In [ ]:
df_students = df_eco101[
    ["student_id", "full_name", "gender", "program_code", "country_of_birth", "result"]
].copy()
df_students.shape

### 6. Remove duplicate rows

The original extract has a quirk: some students have two rows for the same
attempt, one per phone number on file. Now that `phone` is no longer one of our
columns, those rows are exact duplicates of each other.

Print `len(students)`, drop duplicates with `drop_duplicates()`, and print the
new length so you can see how many rows were removed.

In [ ]:
before = len(df_students)
df_students = df_students.drop_duplicates()
after = len(df_students)
print(f"Rows before: {before}, after: {after}, removed: {before - after}")

### 7. Write the student list to CSV

Save `students` to `data/eco101_students.csv` with `to_csv()`. Pass
`index=False` so pandas doesn't write its row-number index as an extra
column.

In [ ]:
OUT_FILE = DATA_FOLDER / "eco101_students.csv"
df_students.to_csv(OUT_FILE, index=False)
print("Wrote:", OUT_FILE.resolve())

### 8. Read the CSV back in

Read `data/eco101_students.csv` back into a new DataFrame called
`students_from_csv`. Print its shape and `.head()`, and compare its `.dtypes` to
`students.dtypes`.

> Excel and CSV store data differently: Excel keeps each column's type, but CSV
> is plain text — pandas has to *guess* each column's type again when it reads
> the file back in. Check whether anything changed.

In [ ]:
df_students_from_csv = pd.read_csv(OUT_FILE)
print(df_students_from_csv.shape)
display(df_students_from_csv.head())
print(df_students.dtypes)
print(df_students_from_csv.dtypes)

### 9. Wrap-up

In your own words (2–3 sentences): what changed between `df` and `students`
(rows, columns, duplicates), and what — if anything — was different about the
DataFrame after the CSV round-trip compared to before you saved it?